In [1]:
!pip install transformers torch torchvision torchaudio numpy pandas tqdm matplotlib huggingface_hub datasets evaluate scikit-learn accelerate


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys
import os

# Path to the folder you want to add
subfolder_path = os.path.join(os.getcwd(), "roberta_classifier")

# Add it to sys.path
if subfolder_path not in sys.path:
    sys.path.append(subfolder_path)

In [3]:
import json
import os
import torch
import sys
import numpy as np
import pandas as pd
from transformers import (
    AutoModelForSequenceClassification,
)

from roberta_classifier.dataset import prepare_datasets
from roberta_classifier.train import train_binary, evaluate_binary
from roberta_classifier.finetuning import save_results, sampling_tuning
from roberta_classifier.seeding import enforce_reproducibility
from roberta_classifier.result_handling import get_balanced_top_configs

results_dir = "roberta_classifier_results"
model_name = "FacebookAI/xlm-roberta-base"
file_base = "_tuning_metrics.csv"

/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
language = "ko"
oversample_ratio, undersample_ratio = get_balanced_top_configs(f"{results_dir}/{language}{file_base}")

enforce_reproducibility(42)

# ==== PREPARE DATASETS ====
train_set, val_set, test_sets, tokenizer = prepare_datasets(model_name, oversample_ratio, undersample_ratio, language)

# ==== MODEL ====
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
)

# ==== TRAIN MODEL ====
model, tokenizer, epoch_history = train_binary(model, train_set, val_set, tokenizer, 1, results_dir)

# ==== EVALUATE MODEL ====
eval_results = evaluate_binary(model, tokenizer, test_sets[language])

# ==== SAVE RESULTS & MODEL ====
save_results(results_dir, language, epoch_history, eval_results)

=== Top Balanced Sampling Configurations ===
╒═════════════════╤══════════════════╤════════════╤═════════════════╤══════════════════╤══════════╤═════════════════╕
│   over_sampling │   under_sampling │   accuracy │   true_accuracy │   false_accuracy │     loss │   balance_score │
╞═════════════════╪══════════════════╪════════════╪═════════════════╪══════════════════╪══════════╪═════════════════╡
│               1 │              0   │   0.980337 │        0.991098 │         0.789474 │ 0.151025 │        0.961543 │
├─────────────────┼──────────────────┼────────────┼─────────────────┼──────────────────┼──────────┼─────────────────┤
│               1 │              1   │   0.980337 │        0.991098 │         0.789474 │ 0.151025 │        0.961543 │
├─────────────────┼──────────────────┼────────────┼─────────────────┼──────────────────┼──────────┼─────────────────┤
│               1 │              0.5 │   0.966292 │        0.976261 │         0.789474 │ 0.21263  │        0.905296 │
╘══════════

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


53.0
[2025-10-30 15:57:07,748] - [INFO] - Starting training of model (answerability classification).


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Confusion Matrix
53,No log,0.670633,0.629630,0.993289,0.624473,0.766839,"[5, 1, 89, 148]"
106,No log,0.147470,0.946502,0.995575,0.949367,0.971922,"[5, 1, 12, 225]"
159,No log,0.142810,0.967078,1.000000,0.966245,0.982833,"[6, 0, 8, 229]"
212,No log,0.154387,0.971193,1.000000,0.970464,0.985011,"[6, 0, 7, 230]"
265,No log,0.167291,0.971193,1.000000,0.970464,0.985011,"[6, 0, 7, 230]"
318,No log,0.214993,0.962963,1.000000,0.962025,0.980645,"[6, 0, 9, 228]"
371,No log,0.223312,0.967078,1.000000,0.966245,0.982833,"[6, 0, 8, 229]"
424,No log,0.172267,0.975309,1.000000,0.974684,0.987179,"[6, 0, 6, 231]"
477,No log,0.189064,0.975309,1.000000,0.974684,0.987179,"[6, 0, 6, 231]"
530,0.223600,0.176857,0.975309,1.000000,0.974684,0.987179,"[6, 0, 6, 231]"


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84v

[2025-10-30 16:03:51,020] - [INFO] - Training completed.
[2025-10-30 16:03:51,031] - [INFO] - Running evaluation...


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[2025-10-30 16:03:51,405] - [INFO] - Evaluation Results:
[2025-10-30 16:03:51,406] - [INFO] - eval_loss: 1.2011722326278687
[2025-10-30 16:03:51,406] - [INFO] - eval_model_preparation_time: 0.0015
[2025-10-30 16:03:51,406] - [INFO] - eval_accuracy: 0.85
[2025-10-30 16:03:51,406] - [INFO] - eval_precision: 0.85
[2025-10-30 16:03:51,406] - [INFO] - eval_recall: 1.0
[2025-10-30 16:03:51,407] - [INFO] - eval_f1: 0.918918918918919
[2025-10-30 16:03:51,407] - [INFO] - eval_confusion_matrix: [0, 3, 0, 17]
[2025-10-30 16:03:51,407] - [INFO] - eval_runtime: 0.3716
[2025-10-30 16:03:51,407] - [INFO] - eval_samples_per_second: 53.817
[2025-10-30 16:03:51,407] - [INFO] - eval_steps_per_second: 8.073
[2025-10-30 16:03:51,416] - [INFO] - Saved per-epoch metrics at: roberta_classifier_results/ko_train_metrics_per_epoch.csv
[2025-10-30 16:03:51,612] - [INFO] - Saved test metrics at: roberta_classifier_results/ko_test_evaluation_metrics.json


In [6]:
language = "ar"
oversample_ratio, undersample_ratio = get_balanced_top_configs(f"{results_dir}/{language}{file_base}")

enforce_reproducibility(42)

# ==== PREPARE DATASETS ====
train_set, val_set, test_sets, tokenizer = prepare_datasets(model_name, oversample_ratio, undersample_ratio, language)

# ==== MODEL ====
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
)

# ==== TRAIN MODEL ====
model, tokenizer, epoch_history = train_binary(model, train_set, val_set, tokenizer, 1, results_dir)

# ==== EVALUATE MODEL ====
eval_results = evaluate_binary(model, tokenizer, test_sets[language])

# ==== SAVE RESULTS & MODEL ====
save_results(results_dir, language, epoch_history, eval_results)

=== Top Balanced Sampling Configurations ===
╒═════════════════╤══════════════════╤════════════╤═════════════════╤══════════════════╤═══════════╤═════════════════╕
│   over_sampling │   under_sampling │   accuracy │   true_accuracy │   false_accuracy │      loss │   balance_score │
╞═════════════════╪══════════════════╪════════════╪═════════════════╪══════════════════╪═══════════╪═════════════════╡
│            0.5  │              0.5 │   0.983133 │        0.983471 │         0.980769 │ 0.0947132 │        1        │
├─────────────────┼──────────────────┼────────────┼─────────────────┼──────────────────┼───────────┼─────────────────┤
│            1    │              0.5 │   0.980723 │        0.983471 │         0.961538 │ 0.108005  │        0.987334 │
├─────────────────┼──────────────────┼────────────┼─────────────────┼──────────────────┼───────────┼─────────────────┤
│            0.25 │              0   │   0.978313 │        0.980716 │         0.961538 │ 0.0983243 │        0.972187 │
╘══

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


25.0
[2025-10-30 16:03:55,421] - [INFO] - Starting training of model (answerability classification).


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Confusion Matrix
25,No log,0.743257,0.101562,0.000000,0.000000,0.000000,"[26, 0, 230, 0]"
50,No log,0.675774,0.753906,0.966480,0.752174,0.845966,"[20, 6, 57, 173]"
75,No log,0.893155,0.492188,0.990196,0.439130,0.608434,"[25, 1, 129, 101]"
100,No log,0.165872,0.960938,0.995495,0.960870,0.977876,"[25, 1, 9, 221]"
125,No log,0.151052,0.960938,0.995495,0.960870,0.977876,"[25, 1, 9, 221]"
150,No log,0.220869,0.960938,0.995495,0.960870,0.977876,"[25, 1, 9, 221]"
175,No log,0.226342,0.960938,0.995495,0.960870,0.977876,"[25, 1, 9, 221]"
200,No log,0.235476,0.960938,0.995495,0.960870,0.977876,"[25, 1, 9, 221]"
225,No log,0.238729,0.960938,0.995495,0.960870,0.977876,"[25, 1, 9, 221]"
250,No log,0.244509,0.960938,0.995495,0.960870,0.977876,"[25, 1, 9, 221]"


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MP

[2025-10-30 16:08:22,985] - [INFO] - Training completed.
[2025-10-30 16:08:22,999] - [INFO] - Running evaluation...


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[2025-10-30 16:08:23,274] - [INFO] - Evaluation Results:
[2025-10-30 16:08:23,274] - [INFO] - eval_loss: 0.3805469870567322
[2025-10-30 16:08:23,274] - [INFO] - eval_model_preparation_time: 0.0012
[2025-10-30 16:08:23,274] - [INFO] - eval_accuracy: 0.85
[2025-10-30 16:08:23,275] - [INFO] - eval_precision: 0.85
[2025-10-30 16:08:23,275] - [INFO] - eval_recall: 1.0
[2025-10-30 16:08:23,275] - [INFO] - eval_f1: 0.918918918918919
[2025-10-30 16:08:23,275] - [INFO] - eval_confusion_matrix: [0, 3, 0, 17]
[2025-10-30 16:08:23,275] - [INFO] - eval_runtime: 0.2703
[2025-10-30 16:08:23,275] - [INFO] - eval_samples_per_second: 73.991
[2025-10-30 16:08:23,275] - [INFO] - eval_steps_per_second: 11.099
[2025-10-30 16:08:23,279] - [INFO] - Saved per-epoch metrics at: roberta_classifier_results/ar_train_metrics_per_epoch.csv
[2025-10-30 16:08:23,355] - [INFO] - Saved test metrics at: roberta_classifier_results/ar_test_evaluation_metrics.json


In [7]:
language = "te"
oversample_ratio, undersample_ratio = get_balanced_top_configs(f"{results_dir}/{language}{file_base}")

enforce_reproducibility(42)

# ==== PREPARE DATASETS ====
train_set, val_set, test_sets, tokenizer = prepare_datasets(model_name, oversample_ratio, undersample_ratio, language)

# ==== MODEL ====
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
)

# ==== TRAIN MODEL ====
model, tokenizer, epoch_history = train_binary(model, train_set, val_set, tokenizer, 4, results_dir)

# ==== EVALUATE MODEL ====
eval_results = evaluate_binary(model, tokenizer, test_sets[language])

# ==== SAVE RESULTS & MODEL ====
save_results(results_dir, language, epoch_history, eval_results)

=== Top Balanced Sampling Configurations ===
╒═════════════════╤══════════════════╤════════════╤═════════════════╤══════════════════╤══════════╤═════════════════╕
│   over_sampling │   under_sampling │   accuracy │   true_accuracy │   false_accuracy │     loss │   balance_score │
╞═════════════════╪══════════════════╪════════════╪═════════════════╪══════════════════╪══════════╪═════════════════╡
│             0.5 │              0   │   0.796875 │        0.945017 │         0.333333 │ 0.980083 │        0.695568 │
├─────────────────┼──────────────────┼────────────┼─────────────────┼──────────────────┼──────────┼─────────────────┤
│             0.5 │              1   │   0.796875 │        0.945017 │         0.333333 │ 0.980083 │        0.695568 │
├─────────────────┼──────────────────┼────────────┼─────────────────┼──────────────────┼──────────┼─────────────────┤
│             1   │              0.5 │   0.710938 │        0.810997 │         0.397849 │ 0.927126 │        0.663236 │
╘══════════

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


88.0
[2025-10-30 16:08:27,097] - [INFO] - Starting training of model (answerability classification).


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Confusion Matrix
88,No log,0.225569,0.941176,0.976744,0.961832,0.969231,"[2, 3, 5, 126]"
176,No log,0.646218,0.852941,0.982609,0.862595,0.918699,"[3, 2, 18, 113]"
264,No log,0.549073,0.941176,0.976744,0.961832,0.969231,"[2, 3, 5, 126]"
352,No log,0.329722,0.977941,0.977612,1.000000,0.988679,"[2, 3, 0, 131]"
440,No log,0.447395,0.970588,0.970370,1.000000,0.984962,"[1, 4, 0, 131]"
528,0.169200,0.422321,0.970588,0.970370,1.000000,0.984962,"[1, 4, 0, 131]"
616,0.169200,0.425584,0.970588,0.970370,1.000000,0.984962,"[1, 4, 0, 131]"
704,0.169200,0.369036,0.970588,0.970370,1.000000,0.984962,"[1, 4, 0, 131]"
792,0.169200,0.364963,0.977941,0.977612,1.000000,0.988679,"[2, 3, 0, 131]"
880,0.169200,0.360909,0.977941,0.977612,1.000000,0.988679,"[2, 3, 0, 131]"


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84v

[2025-10-30 19:39:10,072] - [INFO] - Training completed.
[2025-10-30 19:39:10,084] - [INFO] - Running evaluation...


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[2025-10-30 19:39:10,375] - [INFO] - Evaluation Results:
[2025-10-30 19:39:10,376] - [INFO] - eval_loss: 1.309386134147644
[2025-10-30 19:39:10,376] - [INFO] - eval_model_preparation_time: 0.0014
[2025-10-30 19:39:10,376] - [INFO] - eval_accuracy: 0.85
[2025-10-30 19:39:10,376] - [INFO] - eval_precision: 0.85
[2025-10-30 19:39:10,376] - [INFO] - eval_recall: 1.0
[2025-10-30 19:39:10,376] - [INFO] - eval_f1: 0.918918918918919
[2025-10-30 19:39:10,377] - [INFO] - eval_confusion_matrix: [0, 3, 0, 17]
[2025-10-30 19:39:10,377] - [INFO] - eval_runtime: 0.2874
[2025-10-30 19:39:10,377] - [INFO] - eval_samples_per_second: 69.581
[2025-10-30 19:39:10,377] - [INFO] - eval_steps_per_second: 10.437
[2025-10-30 19:39:10,385] - [INFO] - Saved per-epoch metrics at: roberta_classifier_results/te_train_metrics_per_epoch.csv
[2025-10-30 19:39:10,463] - [INFO] - Saved test metrics at: roberta_classifier_results/te_test_evaluation_metrics.json
